# ShuffleNetV2 0.5x

This standalone notebook trains **ShuffleNetV2 0.5x**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.
Enable `COPY_DATASET_TO_WORKING` in the setup cell to optionally copy the validated dataset to Kaggle working storage for faster I/O.


In [ ]:
import os
from pathlib import Path

def find_dataset_root(input_root='/kaggle/input'):
    candidates = []
    input_root = Path(input_root)
    for root, directories, _ in os.walk(input_root):
        root_path = Path(root)
        if len(root_path.relative_to(input_root).parts) > 6:
            directories[:] = []
            continue
        if {'train', 'val'}.issubset(directories):
            candidates.append(root_path)
            directories[:] = []
    if not candidates:
        raise FileNotFoundError(
            f'No dataset containing train and val folders was found under {input_root}. ' 
            'Attach the leaf-disease dataset through Kaggle Add Input.'
        )
    def rank(path):
        class_count = sum(child.is_dir() for child in (path / 'train').iterdir())
        return (class_count == 72, class_count, len(path.parts))
    return str(max(candidates, key=rank))

DATASET_BASE = find_dataset_root()
print('Selected dataset root:', DATASET_BASE)
for split in ('train', 'val', 'test'):
    split_path = Path(DATASET_BASE) / split
    if split_path.is_dir():
        print(f'  OK {split}: {sum(child.is_dir() for child in split_path.iterdir())} classes')
    else:
        print(f'  MISSING {split} (not required for training)')

In [ ]:
import os
import sys

ARCH = "shufflenetv2_05"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
KAGGLE_INPUT_ROOT = "/kaggle/input"
COPY_DATASET_TO_WORKING = False
WORKING_DATASET_ROOT = "/kaggle/working/leaf_dataset/processed"
OUTPUT_BASE = f"/kaggle/working/leaf_models/{ARCH}"
print("Configured:", ARCH)
print("Dataset search root:", KAGGLE_INPUT_ROOT)
print("Outputs:", OUTPUT_BASE)
print("Copy dataset to Kaggle working storage:", COPY_DATASET_TO_WORKING)


## Environment and repository setup

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git'
BRANCH = 'main'
repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
required_paths = [
    'requirements.txt',
    'training/train.py',
    'training/evaluate.py',
    'quantization/post_training_quant.py',
    'quantization/evaluate_tflite.py',
    'quantization/qat.py',
    'training/config_shufflenetv2_05.yaml',
]

repo_is_complete = all(
    os.path.exists(os.path.join(repo_dir, path)) for path in required_paths
)
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, repo_dir], check=True)
elif not repo_is_complete:
    raise RuntimeError(
        'The existing repo directory is incomplete. Restart the Kaggle session or '
        'remove only /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System, then rerun this cell.'
    )
else:
    print('Complete repository already exists:', repo_dir)

if not all(os.path.exists(os.path.join(repo_dir, path)) for path in required_paths):
    raise FileNotFoundError('Repository clone is missing a required training file.')

print('Repository branch ready:', BRANCH)
print('Repository ready:', repo_dir)

## Install project dependencies

In [ ]:
%pip install -q -r '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/requirements.txt'
%pip install -q tensorflow-model-optimization
print('Dependencies installed successfully!')

In [ ]:
import os
import shutil

os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TFHUB_CACHE_DIR"] = "/kaggle/working/tfhub_modules"
shutil.rmtree(os.environ["TFHUB_CACHE_DIR"], ignore_errors=True)
os.makedirs(os.environ["TFHUB_CACHE_DIR"], exist_ok=True)
print("TF_USE_LEGACY_KERAS:", os.environ["TF_USE_LEGACY_KERAS"])
print("TF Hub cache:", os.environ["TFHUB_CACHE_DIR"])

## Configure paths and detect classes

In [ ]:
import os
import yaml

repo_dir = '/kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System'
cfg_path = os.path.join(repo_dir, 'training', 'config_shufflenetv2_05.yaml')

if 'dataset_base' not in globals():
    dataset_base = DATASET_BASE

output_base = OUTPUT_BASE
os.makedirs(output_base, exist_ok=True)

with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['data_dir'] = dataset_base
cfg['dataset']['train_dir'] = '{}/train'.format(dataset_base)
cfg['dataset']['val_dir'] = '{}/val'.format(dataset_base)
cfg['dataset']['test_dir'] = '{}/test'.format(dataset_base)

train_dir = cfg['dataset']['train_dir']
if not os.path.exists(train_dir):
    raise FileNotFoundError('Train directory not found: {}'.format(train_dir))

num_classes = len([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])
cfg['model']['num_classes'] = num_classes

# Keep all training hyperparameters in config_shufflenetv2_05.yaml.
# This cell only supplies Kaggle paths and the discovered class count.

cfg['export']['save_dir'] = output_base
cfg['export']['sync_flutter_labels'] = False

cfg['callbacks']['tensorboard']['enabled'] = False
cfg['callbacks']['csv_logger']['filename'] = '{}/training_log_shufflenetv2_05.csv'.format(output_base)

with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Configuration updated for Kaggle')
print('Training schedule: {} frozen epochs + {} fine-tune epochs'.format(
    cfg['training']['unfreeze_epoch'],
    cfg['training']['epochs'] - cfg['training']['unfreeze_epoch'],
))
print('Fine-tune from layer: {}; LR: {}; clipnorm: {}'.format(
    cfg['training']['unfreeze_from_layer'],
    cfg['training']['fine_tune_learning_rate'],
    cfg['optimizer'].get('clipnorm'),
))
print('Classes: {}'.format(cfg['model']['num_classes']))
print('Train dataset: {}'.format(cfg['dataset']['train_dir']))
print('Models save to: {}'.format(cfg['export']['save_dir']))
print('TensorBoard logs: DISABLED (saves space)')

## Train the model

In [ ]:
%cd /kaggle/working/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System/training

import os
import subprocess
import sys
import yaml

# Use a conservative, single-T4 runtime even if the cloned repository has
# newer defaults. The prior Kaggle process crashed natively (-11) during its
# first mirrored/XLA step, so this deliberately avoids both paths.
config_path = 'config_shufflenetv2_05.yaml'
with open(config_path, encoding='utf-8') as config_file:
    training_config = yaml.safe_load(config_file)
training_config.setdefault('dataset', {})['batch_size'] = 32
training_config['dataset']['batch_size_fallbacks'] = [32, 16, 8]
training_config['dataset']['auto_batch_fallback'] = False
training_config.setdefault('performance', {})['steps_per_execution'] = 1
training_config['performance']['distribution'] = 'single'
training_config['performance']['multi_gpu'] = False
# The shared config retains the XLA switch for compatible systems.
training_config['performance']['jit_compile'] = False
runtime_config_path = 'config_shufflenetv2_05_kaggle_t4.yaml'
with open(runtime_config_path, 'w', encoding='utf-8') as config_file:
    yaml.safe_dump(training_config, config_file, sort_keys=False)
print(
    'Kaggle runtime settings:',
    f"batch_size={training_config['dataset']['batch_size']}, "
    f"distribution={training_config['performance']['distribution']}, "
    f"steps_per_execution={training_config['performance']['steps_per_execution']}, "
    f"jit_compile={training_config['performance']['jit_compile']}",
)

training_command = [
    sys.executable, '-X', 'faulthandler', '-u', 'train.py',
    '--config', runtime_config_path,
]
runtime_environment = dict(os.environ)
runtime_environment['CUDA_VISIBLE_DEVICES'] = '0'
runtime_environment['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
training_result = subprocess.run(
    training_command, check=False, env=runtime_environment
)
if training_result.returncode != 0:
    raise RuntimeError(
        f'Training process failed with exit code {training_result.returncode}'
    )

## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(OUTPUT_BASE)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(OUTPUT_BASE, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", cfg_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
quant_dir = os.path.join(OUTPUT_BASE, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--evaluate",
    "--test_data", cfg["dataset"]["test_dir"],
    "--arch", ARCH,
], check=True)
print("Exported:", dynamic_path, int8_path)

## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(OUTPUT_BASE, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", OUTPUT_BASE)
print("Archive:", archive_path)

## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.